# Knowing When the Model Doesn't Know
## Notebook 01: Data exploration and baseline CNN

CSCI 6366 Neural Networks and Deep Learning, final project.

This notebook covers the **initial stages** of the project:

1. Load PneumoniaMNIST (main dataset) and BreastMNIST / OCTMNIST (out-of-distribution sets).
2. Explore the data: sizes, class balance, sample images.
3. Define a small CNN with dropout layers (needed later for MC Dropout).
4. Train the baseline model and check its accuracy.
5. Take a first look at softmax confidence on correct vs. wrong predictions.

Runs on Google Colab (Runtime > Change runtime type > GPU). A CPU also works, just slower.

In [ ]:
# Install the MedMNIST package (needed on Colab)
!pip install -q medmnist

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt
import medmnist
from medmnist import PneumoniaMNIST, BreastMNIST, OCTMNIST

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
DATA_ROOT = "data"

import os
os.makedirs(DATA_ROOT, exist_ok=True)
print(f"Device: {DEVICE} | medmnist version: {medmnist.__version__}")

## 1. Load the datasets

All images are 28x28 grayscale. We scale pixels to [0, 1] and then normalize to roughly [-1, 1].

| Dataset | Image type | Role |
|---|---|---|
| PneumoniaMNIST | Chest X-ray (normal vs. pneumonia) | Train / validate / test |
| BreastMNIST | Breast ultrasound | Out-of-distribution (never trained on) |
| OCTMNIST | Retinal OCT scan | Out-of-distribution (never trained on) |

In [ ]:
transform = T.Compose([
    T.ToTensor(),                      # [0, 255] -> [0, 1]
    T.Normalize(mean=[0.5], std=[0.5]) # [0, 1]  -> [-1, 1]
])

def load_split(dataset_class, split):
    return dataset_class(split=split, transform=transform, download=True, root=DATA_ROOT)

train_set = load_split(PneumoniaMNIST, "train")
val_set   = load_split(PneumoniaMNIST, "val")
test_set  = load_split(PneumoniaMNIST, "test")

ood_breast = load_split(BreastMNIST, "test")
ood_oct    = load_split(OCTMNIST, "test")

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

CLASS_NAMES = ["normal", "pneumonia"]

## 2. Explore the data

In [ ]:
def class_counts(dataset):
    labels = dataset.labels.flatten()
    return {name: int((labels == i).sum()) for i, name in enumerate(CLASS_NAMES)}

print(f"{'Split':<8}{'Total':>8}{'Normal':>10}{'Pneumonia':>12}{'% Pneumonia':>14}")
for name, ds in [("train", train_set), ("val", val_set), ("test", test_set)]:
    c = class_counts(ds)
    total = len(ds)
    print(f"{name:<8}{total:>8}{c['normal']:>10}{c['pneumonia']:>12}{100 * c['pneumonia'] / total:>13.1f}%")

print(f"\nOOD sets: BreastMNIST test = {len(ood_breast)} images, OCTMNIST test = {len(ood_oct)} images")

Check the **% Pneumonia** column: PneumoniaMNIST is imbalanced, with roughly three pneumonia images
for every normal one. This matters later, because a model can look accurate just by leaning toward
"pneumonia," so we also report results for each class separately.

In [ ]:
def show_examples(dataset, title, n=8, label_names=None):
    fig, axes = plt.subplots(1, n, figsize=(n * 1.6, 2.1))
    for ax, idx in zip(axes, range(n)):
        img, label = dataset[idx]
        ax.imshow(img.squeeze(0), cmap="gray")
        ax.set_title(label_names[int(np.asarray(label).item())] if label_names else "", fontsize=8)
        ax.axis("off")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()

show_examples(train_set, "PneumoniaMNIST (training data)", label_names=CLASS_NAMES)
show_examples(ood_breast, "BreastMNIST (out-of-distribution)")
show_examples(ood_oct, "OCTMNIST (out-of-distribution)")

## 3. Model: a small CNN with dropout

Standard architectures like ResNet do not include dropout layers. We build our own small CNN so
dropout sits after every convolution block and before the classifier. During normal testing,
dropout is turned off. For **Monte Carlo Dropout** (next notebook) we keep it on at test time and
run each image many times; the spread of the predictions is the uncertainty.

In [ ]:
class DropoutCNN(nn.Module):
    """Three conv blocks (conv -> batchnorm -> ReLU -> maxpool -> dropout) and a classifier with dropout."""

    def __init__(self, num_classes=2, dropout_p=0.3):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Dropout(dropout_p),
            )

        self.features = nn.Sequential(
            conv_block(1, 32),    # 28x28 -> 14x14
            conv_block(32, 64),   # 14x14 -> 7x7
            conv_block(64, 128),  # 7x7   -> 3x3
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = DropoutCNN().to(DEVICE)
print(model)
print(f"\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Train the baseline model

In [ ]:
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

def run_epoch(model, loader, train):
    model.train(train)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.squeeze(1).long().to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            count += len(labels)
    return total_loss / count, correct / count

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc, best_state = 0.0, None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, train=False)
    for key, value in zip(history, [train_loss, val_loss, train_acc, val_acc]):
        history[key].append(value)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f"Epoch {epoch:2d} | train loss {train_loss:.3f} acc {train_acc:.3f} | val loss {val_loss:.3f} acc {val_acc:.3f}")

model.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_val_acc:.3f} (weights restored)")

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.8))
ax_loss.plot(epochs, history["train_loss"], label="train")
ax_loss.plot(epochs, history["val_loss"], label="validation")
ax_loss.set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
ax_acc.plot(epochs, history["train_acc"], label="train")
ax_acc.plot(epochs, history["val_acc"], label="validation")
ax_acc.set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
for ax in (ax_loss, ax_acc):
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Test results and first look at softmax confidence

In [ ]:
@torch.no_grad()
def predict_softmax(model, loader):
    """Returns softmax probabilities and true labels for a whole dataset (dropout OFF)."""
    model.eval()
    all_probs, all_labels = [], []
    for images, labels in loader:
        all_probs.append(F.softmax(model(images.to(DEVICE)), dim=1).cpu())
        all_labels.append(labels.squeeze(1))
    return torch.cat(all_probs).numpy(), torch.cat(all_labels).numpy()

test_probs, test_labels = predict_softmax(model, test_loader)
test_preds = test_probs.argmax(1)
confidence = test_probs.max(1)
is_correct = test_preds == test_labels

print(f"Test accuracy: {is_correct.mean():.3f}")
for i, name in enumerate(CLASS_NAMES):
    mask = test_labels == i
    print(f"  Recall on {name:<9}: {(test_preds[mask] == i).mean():.3f}  ({mask.sum()} images)")
print(f"\nMean confidence when correct: {confidence[is_correct].mean():.3f}")
print(f"Mean confidence when wrong:   {confidence[~is_correct].mean():.3f}")

In [ ]:
bins = np.linspace(0.5, 1.0, 21)
plt.figure(figsize=(7, 3.8))
plt.hist(confidence[is_correct], bins=bins, alpha=0.7, label=f"correct (n={is_correct.sum()})")
plt.hist(confidence[~is_correct], bins=bins, alpha=0.7, label=f"wrong (n={(~is_correct).sum()})")
plt.xlabel("Softmax confidence (max probability)")
plt.ylabel("Number of test images")
plt.title("Baseline: how confident is the model on right vs. wrong answers?")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

If many wrong predictions still sit near confidence 1.0, softmax alone is not a reliable warning
signal. That is the problem the rest of the project addresses.

### Quick check: confidence on images the model has never seen

These are ultrasound and retinal images, so the model's "normal vs. pneumonia" answer is
meaningless for them. Ideally its confidence should drop.

In [ ]:
def ood_confidence(dataset):
    probs, _ = predict_softmax(model, DataLoader(dataset, batch_size=BATCH_SIZE))
    return probs.max(1)

conf_breast = ood_confidence(ood_breast)
conf_oct = ood_confidence(ood_oct)

print(f"{'Data':<34}{'Mean confidence':>16}")
print(f"{'PneumoniaMNIST test (familiar)':<34}{confidence.mean():>16.3f}")
print(f"{'BreastMNIST (never seen)':<34}{conf_breast.mean():>16.3f}")
print(f"{'OCTMNIST (never seen)':<34}{conf_oct.mean():>16.3f}")

plt.figure(figsize=(7, 3.8))
for values, label in [(confidence, "PneumoniaMNIST (familiar)"), (conf_breast, "BreastMNIST (OOD)"), (conf_oct, "OCTMNIST (OOD)")]:
    plt.hist(values, bins=bins, alpha=0.55, density=True, label=label)
plt.xlabel("Softmax confidence")
plt.ylabel("Density")
plt.title("Baseline confidence on familiar vs. never-seen images")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Save the baseline model

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/baseline_seed42.pt")
print("Saved checkpoints/baseline_seed42.pt")

## Next steps

- **MC Dropout:** keep dropout on at test time, run each image about 30 times, use the spread as uncertainty.
- **Deep Ensemble:** train this same CNN with 5 different seeds and average the predictions.
- **Temperature scaling:** fit one temperature on the validation set to recalibrate the softmax.
- **Distribution shift:** add noise, blur, and lower contrast to the test images at several severity levels.
- **Evaluation:** ECE, NLL, Brier score, OOD AUROC, accuracy vs. coverage, and reliability diagrams.